In [5]:
import pandas as pd
import json
import numpy as np
import os

In [7]:
import re
def get_error_type(text):
    # Regex to match and extract 'AssertionError'
    match = re.search(r"type:\s*<class '([\w.]+)'>", text)
    
    if match:
        extracted_error = match.group(1)
        #if extracted_error == "AssertionError":
            #return "passed"
        #else:
        return extracted_error
    else:
        return text

In [8]:
import pickle
# Load the pipeline
with open('pipeline_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)

In [88]:
# Function to count passed
def count_passed(df):
    return len(df[df['execute_message'] == "passed"])
    
# Function to count assertion error
def count_assertion(df, bench='odex'):
    if bench == 'ds':
        return len(df[df['error_type'].str.contains("AssertionError")])
    elif bench == 'odex':
        return len(df[df['execute_message'] == "failed: "])
        
# Function to count original IFS
def count_ifs(df):
    # List of values to check
    values_list = ['TypeError', 'AttributeError', 'ModuleNotFoundError', 'ImportError']
    
    # Count rows where 'col1' contains values in values_list
    count = df['error_type'].isin(values_list).sum()
    return count

In [71]:
file_names_odex = ["results/odex/answers/claude-3-opus-20240229-0-answers.jsonl", 
              "results/odex/answers/gpt-3.5-turbo-0125-0-answers.jsonl",
              "results/odex/answers/gpt-4-0613-0-answers.jsonl",
              "results/odex/answers/granite-code-latest-0-answers.jsonl"]

In [80]:
file_names_ds = ["results/errors/claude-3-opus-20240229-0-errors.jsonl", 
                "results/errors/gpt-3.5-turbo-0125-0-errors.jsonl", 
              "results/errors/gpt-4-0613-0-errors.jsonl",
              "results/errors/granite-code-latest-no-grammar-0-errors.jsonl"]

In [76]:
for file in file_names_odex:
    print(file)
    file_df = pd.read_json(file,lines=True)
    file_df = file_df.rename(columns={'result': 'execute_message'})
    assertion_errors = count_assertion(file_df, 'odex')
    passed_rows = count_passed(file_df)
    file_df["error_type"] = pipeline.predict(file_df["execute_message"])
    print(f"ER: {assertion_errors+passed_rows}, passed: {passed_rows}, ifs: {count_ifs(file_df)}")

results/odex/answers/claude-3-opus-20240229-0-answers.jsonl
ER: 338, passed: 213, ifs: 103
results/odex/answers/gpt-3.5-turbo-0125-0-answers.jsonl
ER: 301, passed: 181, ifs: 80
results/odex/answers/gpt-4-0613-0-answers.jsonl
ER: 288, passed: 178, ifs: 69
results/odex/answers/granite-code-latest-0-answers.jsonl
ER: 234, passed: 91, ifs: 121


In [89]:
for file in file_names_ds:
    print(file)
    file_df = pd.read_json(file,lines=True)
    file_df = file_df.rename(columns={'result': 'execute_message'})
    file_df['error_type'] = file_df['execute_message'].apply(get_error_type)
    assertion_errors = count_assertion(file_df, 'ds')
    passed_rows = count_passed(file_df)
    print(f"ER: {assertion_errors+passed_rows}, passed: {passed_rows}, ifs: {count_ifs(file_df)}")

results/errors/claude-3-opus-20240229-0-errors.jsonl
ER: 731, passed: 427, ifs: 91
results/errors/gpt-3.5-turbo-0125-0-errors.jsonl
ER: 732, passed: 366, ifs: 97
results/errors/gpt-4-0613-0-errors.jsonl
ER: 768, passed: 477, ifs: 81
results/errors/granite-code-latest-no-grammar-0-errors.jsonl
ER: 336, passed: 89, ifs: 114


In [ ]:
# Files to fix

In [2]:
# Enum completion_id
file_names_fix_odex_granite = ["results/classifier/annotated/granite-code-latest-general-0-answers.csv",
"results/classifier/annotated/granite-code-latest-union-0-answers.csv",
"results/classifier/annotated/granite-code-latest-mypy-0-answers.csv",
"results/classifier/annotated/granite-code-latest-pyright-0-answers.csv"]

file_names_fix_odex_o3 = ["results/classifier/annotated/o3-mini-general-0-answers.csv",
"results/classifier/annotated/o3-mini-union-0-answers.csv",
"results/classifier/annotated/o3-mini-mypy-0-answers.csv",
"results/classifier/annotated/o3-mini-pyright-0-answers.csv"]

In [62]:
ds1000 = pd.read_csv('DS1000_detector_al_errors.csv')
odex = pd.read_csv('Odex_detector_al_errors.csv')

In [24]:
ds1000['error_type'] = ds1000['execute_message'].apply(get_error_type)
ds1000_error = ds1000[ds1000['execute']==0]
ds1000_error['enum_id'] = ds1000_error.groupby('completion_id').cumcount()+ 1

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_1872/3349914892.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds1000_error['enum_id'] = ds1000_error.groupby('completion_id').cumcount()+ 1


In [63]:
odex["error_type"] = pipeline.predict(odex["execute_message"])
odex_error = odex[odex['execute']==0]
odex_error[['model', 'id']] = odex_error['completion_id'].str.extract(r"(.+)[-_](?:0_)?(\d+)\.py")
# Enumerate occurrences of the same 'after' value
odex_error['enum_id'] = odex_error.groupby('id').cumcount()+ 1

In [140]:
df_o3_odex = pd.DataFrame()
for file in file_names_fix_odex_granite:
    odex_csv = pd.read_csv(file)
    odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1
    odex_csv['id'] = odex_csv['id'].astype(str)
    merge_df = pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id','error_type']], on=['id','enum_id'], how='left')
    # List of prediction values to keep
    #filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df_repair = merge_df[merge_df['prediction'].isin(filter_values)]
    gdf = filtered_df_repair.groupby(['model','prediction'])['id'].count().reset_index()
    gdf = gdf.groupby(['model'])['id'].sum().reset_index()
    # Filter total
    filtered_df_total = merge_df[merge_df['error_type'].isin(filter_values)]
    gdf_total = filtered_df_total.groupby(['model','error_type'])['id'].count().reset_index()
    gdf_total = gdf_total.groupby(['model'])['id'].sum().reset_index()
    merge_df_new = pd.merge(gdf_total, gdf, on=['model'], how='left', suffixes=("_before","_after"))
    merge_df_new['label'] = file
    merge_df_new['bench'] = 'odex'
    df_o3_odex = pd.concat([df_o3_odex, merge_df_new])
df_o3_odex.to_csv('granite_general_odex_ifs.csv', index=False)

In [105]:
df_o3_odex = pd.DataFrame()
for file in file_names_fix_odex_o3:
    odex_csv = pd.read_csv(file)
    odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1
    odex_csv['id'] = odex_csv['id'].astype(str)
    merge_df = pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id']], on=['id','enum_id'], how='left')
    # List of prediction values to keep
    #filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
     filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['prediction'].isin(filter_values)]
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','prediction'])['id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'odex'
    df_o3_odex = pd.concat([df_o3_odex, gdf])
df_o3_odex.to_csv('o3_mini_general_odex.csv', index=False)

In [106]:
df_o3_odex = pd.DataFrame()
for file in file_names_fix_odex_granite:
    odex_csv = pd.read_csv(file)
    odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1
    odex_csv['id'] = odex_csv['id'].astype(str)
    merge_df = pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id']], on=['id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['prediction'].isin(filter_values)]
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','prediction'])['id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'granite'
    df_o3_odex = pd.concat([df_o3_odex, gdf])
df_o3_odex.to_csv('o3_mini_general_granite.csv', index=False)

In [3]:
# Enum id
file_names_fix_ds_o3 = ['results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl',
'results/errors/o3/o3-mini-repair-union-0-errors.jsonl',
'results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl',
'results/errors/o3/o3-mini-repair-general-0-errors.jsonl',]

file_names_fix_ds_granite = ['results/errors/ibm/repair/granite-code-latest-repair-pyright-0-errors.jsonl',
'results/errors/ibm/repair/granite-code-latest-repair-union-0-errors.jsonl',
'results/errors/ibm/repair/granite-code-latest-repair-mypy-0-errors.jsonl',
'results/errors/ibm/repair/granite-code-latest-repair-general-0-errors.jsonl',]

In [114]:
odex_csv = pd.read_json('results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl', lines=True)
odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
odex_csv

,passed,result,completion_id,generate_code,score,library,perturbation_type,enum_id
0,False,failed: expected an indented block after funct...,12,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Surface,1
1,False,"failed: 'result', type: <class 'KeyError'>",0,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Origin,1
2,False,"failed: , type: <class 'AssertionError'>",28,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Difficult-Rewrite,1
3,False,failed: expected an indented block after funct...,4,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Surface,1
4,False,failed: expected an indented block after funct...,4,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Surface,2
...,...,...,...,...,...,...,...,...
821,False,failed: name 'confidence_score' is not defined...,993,import torch\nimport copy\nimport sklearn\nfro...,0,Pytorch,Origin,3
822,False,failed: Sizes of tensors must match except in ...,998,import torch\nimport copy\n\n\ndef generate_te...,0,Pytorch,Semantic,1
823,False,"failed: , type: <class 'AssertionError'>",993,import torch\nimport copy\nimport sklearn\nfro...,0,Pytorch,Origin,4
824,False,failed: Sizes of tensors must match except in ...,998,import torch\nimport copy\n\n\ndef generate_te...,0,Pytorch,Semantic,2


In [118]:
df_o3_ds = pd.DataFrame()
for file in file_names_fix_ds_o3:
    odex_csv = pd.read_json(file, lines=True)
    odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
    odex_csv['error_type'] = odex_csv['result'].apply(get_error_type)
    merge_df = pd.merge(odex_csv[['completion_id','error_type','enum_id']],ds1000_error[['model','completion_id','enum_id']], on=['completion_id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['error_type'].isin(filter_values)]
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','error_type'])['completion_id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'ds'
    gdf['model_bench'] = 'o3'
    df_o3_ds = pd.concat([df_o3_ds, gdf])
df_o3_ds.to_csv('o3_mini_general_ds.csv', index=False)

In [120]:
df_o3_granite = pd.DataFrame()
for file in file_names_fix_ds_granite :
    odex_csv = pd.read_json(file, lines=True)
    odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
    odex_csv['error_type'] = odex_csv['result'].apply(get_error_type)
    merge_df = pd.merge(odex_csv[['completion_id','error_type','enum_id']],ds1000_error[['model','completion_id','enum_id']], on=['completion_id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['error_type'].isin(filter_values)]
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','error_type'])['completion_id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'ds'
    gdf['model_bench'] = 'o3'
    df_o3_granite = pd.concat([df_o3_granite, gdf])
df_o3_granite.to_csv('granite_mini_general_ds.csv', index=False)

In [146]:
df_o3_ds = pd.DataFrame()
for file in file_names_fix_ds_granite:
    odex_csv = pd.read_json(file, lines=True)
    odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
    odex_csv['error_type'] = odex_csv['result'].apply(get_error_type)
    merge_df = pd.merge(odex_csv[['completion_id','error_type','enum_id']],ds1000_error[['model','completion_id','enum_id','error_type']], on=['completion_id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df_repair = merge_df[merge_df['error_type_x'].isin(filter_values)]
    gdf = filtered_df_repair.groupby(['model','error_type_x'])['completion_id'].count().reset_index()
    gdf = gdf.groupby(['model'])['completion_id'].sum().reset_index()
    # Filter total
    filtered_df_total = merge_df[merge_df['error_type_y'].isin(filter_values)]
    gdf_total = filtered_df_total.groupby(['model','error_type_y'])['completion_id'].count().reset_index()
    gdf_total = gdf_total.groupby(['model'])['completion_id'].sum().reset_index()
    merge_df_new = pd.merge(gdf_total, gdf, on=['model'], how='left', suffixes=("_before","_after"))
    merge_df_new = pd.merge(gdf_total, gdf, on=['model'], how='left', suffixes=("_before","_after"))
    merge_df_new['label'] = file
    merge_df_new['bench'] = 'odex'
    df_o3_ds = pd.concat([df_o3_ds, merge_df_new])
df_o3_ds.to_csv('granite_general_ds_ifs.csv', index=False)

In [65]:
odex_csv = pd.read_csv('results/classifier/annotated/o3-mini-general-0-answers.csv')
odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1

In [68]:
pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id','error_type']], on=['id','enum_id'], how='left').groupby(['model','prediction']).count()

id  enum_id  error_type
model                    prediction                                  
claude-3-opus-20240229-0 AssertionError       10       10          10
                         AttributeError       23       23          23
                         FileNotFoundError     2        2           2
                         IndexError            2        2           2
                         KeyError             12       12          12
                         ModuleNotFoundError   3        3           3
                         NameError             4        4           4
                         TestCaseError         1        1           1
                         TypeError            41       41          41
                         ValueError           14       14          14
                         passed                7        7           7
                         timed out             4        4           4
gpt-3.5-0125             AssertionError       16       16          16
                         AttributeError       22       22          22
                         FileNotFoundError     3        3           3
                         IndexError            5        5           5
                         KeyError              9        9           9
                         ModuleNotFoundError   5        5           5
                         NameError             3        3           3
                         SyntaxError           1        1           1
                         TestCaseError         1        1           1
                         TypeError            37       37          37
                         ValueError           16       16          16
                         passed               23       23          23
                         timed out             9        9           9
gpt-4-0613               AssertionError       13       13          13
                         AttributeError       12       12          12
                         IndexError            1        1           1
                         KeyError              6        6           6
                         ModuleNotFoundError   1        1           1
                         NameError             1        1           1
                         TypeError            20       20          20
                         ValueError            4        4           4
                         passed               40       40          40
                         timed out             5        5           5
granite-code-latest-0    AssertionError       12       12          12
                         AttributeError        2        2           2
                         ImportError           1        1           1
                         KeyError              2        2           2
                         ModuleNotFoundError   2        2           2
                         NameError             1        1           1
                         SyntaxError           1        1           1
                         TypeError             9        9           9
                         ValueError            3        3           3
                         passed                8        8           8
                         timed out             3        3           3